# BPE - Sennrich, Haddow & Birch (2016)

*Neural Machine Translation of Rare Words with Subword Units*, ACL 2016.

Paper: `~/personal/knowledge/projects/nlp-to-llm-evolution/sources/sennrich-2016-bpe.pdf`

Tier: Light (2 working days). Started 2026-10-16.


## Pain point: before this paper
1. Limited Vocabulary sizes
2. To handle <UNK> words people used back-off dictionary
3. Problem with this approach is it was not able to handle
  - Compound words i.e word made of multiple words
  - Names, because names are evolving and every name cannot be in dictionary
  - Word Morphology i.e un + happi + ness
4. In all the cases the Translation is done on a smaller Units
5. Why not build at character level, because then sequence will be longer and RNNs will have hard time processing it.

## Middle ground: which words stay whole, which get split?
1. This is decided based on frequency of occurance. So if a word is frequent it will get more updates during training and
   word vector representation will be better. We do not need to break frequent words. But rare words, we can break. Doing so
   will increase the frequency of these pieces and since the split words will be neighbours model will learn to join them.
2. Consistency is important.
3. The goal of paper is zero <UNK>

## Open vocabulary: why BPE never produces UNK
1. Because characters are always present in vocabulary and any new word can be broken down into characters

## Learning BPE: the merge loop
**Input:** a list of words with their counts, e.g. `low x5, lower x2, newest x6, widest x3`.

1. Split every word into characters and add `</w>` at the end: `newest` -> `n e w e s t </w>`.
   `</w>` marks the end of a word, so `est</w>` (word end) and `est` (middle of a word) stay different.
2. Count every pair of adjacent symbols. Each pair gets the count of the word it is in
   (`newest x6` gives every pair inside it +6).
3. Pick the pair with the highest count.
4. Merge that pair everywhere it appears. It is now one symbol.
5. Write the merge down in a list, in order: `e s`, `es t`, ...
6. Repeat steps 2-5 N times. N (number of merges) is a hyperparameter.
   - N = 0 -> pure characters
   - N very large -> every word is one token, back to word level

Pairs are counted only inside a word, never across two words.

**Output:** the ordered list of merges. This list is what we use later to split new words.


## Applying BPE: segmenting an unseen word
Lets say new word is `l o w e s t </w>`
Merge list from training `[e s, es t, est </w>, l o, lo w]`

Spaces separate symbols, so `t </w>` is two symbols and `t</w>` is one.

1. `e s`: first merge is e s, so we will merge e s in the word -> `l o w es t </w>`
2. `es t`: now es and t ko merge krna hai -> `l o w est </w>`
3. `est </w>`: -> `l o w est</w>`
4. `l o`: -> `lo w est</w>`
5. `lo w`: -> `low est</w>`

Final: `low + est</w>`

**How this differs from learning:**
- No counting here. We just replay the merge list in the learned order; if a pair is not in the word, skip that merge.
- Why not run the learning loop on the new word? In a single word every pair has count 1, so all pairs tie and the split would be arbitrary.
  The frequency signal came from the training corpus and is already stored in the merge list.


# Implementation

Blocks 1-6 define everything; blocks 7-8 are the experiments and only call those functions.
Blocks 3-6 are copied as-is from `bahdanau-2014.ipynb` (toy task cells left out).

## 1. Setup

In [ ]:
import heapq
import random
from collections import Counter
from typing import Any

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader

from common.utils.checkpoint import Checkpointer, CheckpointState
from common.utils.trainer import Trainer

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
seed = 42
random.seed(seed)
torch.manual_seed(seed)
print(f"Using device: {device}")

TOK_UNK = "<UNK>"
TOK_EOS = "<EOS>"
TOK_SOS = "<SOS>"
TOK_PAD = "<PAD>"
TOK_UNK_ID = 1
TOK_EOS_ID = 3
TOK_SOS_ID = 2
TOK_PAD_ID = 0

In [ ]:
class PairCountTracker:
    def __init__(self) -> None:
        self.counts = {}
        self.heap = []
        # pairs whose count changed since the last push; pushed once in flush()
        self.dirty = set()

    def set_count(self, key, value):
        if key not in self.counts:
            self.counts[key] = 0
        self.counts[key] += value
        if self.counts[key] <= 0:
            # pair no longer exists anywhere; its heap entries become stale
            del self.counts[key]
            self.dirty.discard(key)
            return
        self.dirty.add(key)

    def flush(self):
        """Push one heap entry per changed pair, with its final count."""
        if not self.heap:
            # first flush: build the heap in one O(n) step
            self.heap = [(-self.counts[key], key) for key in self.dirty]
            heapq.heapify(self.heap)
        else:
            for key in self.dirty:
                heapq.heappush(self.heap, (-self.counts[key], key))
        self.dirty.clear()

    def pop_frequent(self) -> tuple[str, str]:
        self.flush()
        while self.heap:
            neg_val, key = self.heap[0]
            current_val = -neg_val
            if key in self.counts and self.counts[key] == current_val:
                del self.counts[key]
                return (key, current_val)
            heapq.heappop(self.heap)
        return ("", "")
    
    def exists(self, key):
        return key in self.counts

## 2. BPE: learn and apply

In [ ]:
def merge_pair(word, pair):
    """Merge every non-overlapping occurrence of pair in word, left to right."""
    new_word = []
    skip_next = False
    for char_pos in range(len(word)):
        if skip_next:
            # this symbol was already merged into the previous one
            skip_next = False
            continue
        if char_pos+1 < len(word) and (word[char_pos], word[char_pos+1]) == pair:
            new_word.append(word[char_pos] + word[char_pos+1])
            skip_next = True
        else:
            new_word.append(word[char_pos])
    return tuple(new_word)


def word_pairs(word):
    return [(word[i], word[i + 1]) for i in range(len(word) - 1)]


def learn_bpe(word_counts, num_merges):
    pairs = PairCountTracker()
    words = list(word_counts.keys())                  # word_id -> current symbols
    word_freq = [word_counts[word] for word in words]  # word_id -> count
    pair_word_map = {}                                  # pair -> set of word_ids
    for word_id, word in enumerate(words):
        for pair in word_pairs(word):
            pairs.set_count(pair, word_freq[word_id])
            pair_word_map.setdefault(pair, set()).add(word_id)
    merges = []
    for _ in range(num_merges):
        frequent_pair, _ = pairs.pop_frequent()
        if frequent_pair == "":
            # every word is a single symbol, nothing left to merge
            break
        merges.append(frequent_pair)
        # snapshot: the loop below edits pair_word_map
        word_ids = list(pair_word_map.pop(frequent_pair, ()))
        for word_id in word_ids:
            word = words[word_id]
            count = word_freq[word_id]
            new_word = merge_pair(word, frequent_pair)
            # whole-word diff: remove every old pair, add every new pair,
            # so a pair shared by two merged symbols is counted once
            for pair in word_pairs(word):
                if pair != frequent_pair:  # already removed by pop_frequent
                    pairs.set_count(pair, -count)
            for pair in word_pairs(new_word):
                pairs.set_count(pair, count)
            # the index is per word, not per occurrence, so use sets here
            for pair in set(word_pairs(word)) - {frequent_pair}:
                pair_word_map[pair].discard(word_id)
            for pair in set(word_pairs(new_word)):
                pair_word_map.setdefault(pair, set()).add(word_id)
            words[word_id] = new_word
    return merges


def apply_bpe(word, merges):
    for pair in merges:
        if pair[0] in word and pair[1] in word:
            word = merge_pair(word, pair)
    return list(word)

In [ ]:
test = {
    ('l', 'o', 'w', '</w>'): 5, 
    ('l', 'o', 'w', 'e', 'r', '</w>'): 2, 
    ('n', 'e', 'w', 'e', 's', 't', '</w>'): 6, 
    ('w','i', 'd', 'e', 's', 't', '</w>'): 3
} 
merges = learn_bpe(test, 5)
print(merges)

test_1 = {
    ('a', 'b', 'c', '</w>'): 5,
    ('b', 'c', '</w>'): 4
}
merges_1 = learn_bpe(test_1, 3)
print(merges_1)

In [ ]:
print(apply_bpe(('l', 'o', 'w', 'e', 's', 't', '</w>'), merges))

In [ ]:
w = "abc"
a = tuple(w)

In [ ]:
# Segment a whole corpus with BPE. TODO (Vikas)
# Many tokens, far fewer unique words: segment each unique word once and reuse it.
def segment_corpus(sentences, merges):
    """sentences: list of token lists -> list of subword lists."""
    processed = {}
    all_subwords = []
    for sentence in sentences:
        subwords = []
        for word in sentence:
            word_tuple = tuple(word)
            word_tuple = word_tuple + ("</w>",)
            word_tuple_bpe = None
            if word_tuple not in processed:
                word_tuple_bpe = apply_bpe(word_tuple, merges)
                processed[word_tuple] = word_tuple_bpe
            subwords.extend(processed.get(word_tuple, []))
        all_subwords.append(subwords)
    return all_subwords

In [ ]:
# Undo BPE on decoder output before BLEU. TODO (Vikas)
def undo_bpe(subwords):
    """subwords: list of BPE symbols (with </w>) -> list of words."""
    words = []
    start = 0
    for idx, subword in enumerate(subwords):
        if start >= len(subwords):
            break
        if subword.endswith("</w>"):
            piece = subword.replace("</w>", "")
            word_list = subwords[start:idx+1]
            word_list[-1] = piece
            words.append("".join(word_list))
            start = idx+1
    if start < len(subwords):
        words.append("".join(subwords[start:]))
    return words

In [ ]:
undo_bpe(['le</w>', 'ré', 'invent'])

## 3. Data

The `# eos-data` cell must stay the first data cell (`:ColabData` runs it).

In [ ]:
# eos-data: HF WMT14 fr-en
# Pipeline: random sample -> Moses tokenize -> length filter -> cache.
# Colab: full download to VM disk (lost with the VM), tokenized cache on Drive
# (survives). Mac: streaming slice only, never the full download.
# Local DATA_DIR is relative to where nvim was launched; open nvim at the project root.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("sacremoses") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sacremoses"])

from datasets import load_dataset
from sacremoses import MosesTokenizer

try:
    import google.colab  # noqa: F401

    ON_COLAB = True
    HF_CACHE = "/root/colab-data/hf"
    CACHE_DIR = "/content/drive/MyDrive/genesis-lab/data/wmt14_fr_en"
except ImportError:
    ON_COLAB = False
    HF_CACHE = None
    CACHE_DIR = "data/wmt14_fr_en"

# pairs drawn before the length filter; default for load_hf_wmt14_data(n_sample=...)
N_SAMPLE = 2_000_000 if ON_COLAB else 2_000
MAX_LEN = 50  # RNN*-50: both sides at most 50 tokens
DATA_SEED = 42

if ON_COLAB:
    assert os.path.isdir("/content/drive/MyDrive"), "Drive not mounted: run :ColabDrive first"

_moses = {"en": MosesTokenizer(lang="en"), "fr": MosesTokenizer(lang="fr")}


def tokenize_pair(example):
    # escape=False keeps ' and & as they are instead of &apos; / &amp;
    tr = example["translation"]
    return {
        "src": _moses["en"].tokenize(tr["en"], escape=False),
        "tgt": _moses["fr"].tokenize(tr["fr"], escape=False),
    }


def to_lists(rows, max_len):
    src, tgt = [], []
    for row in rows:
        s, t = row["src"], row["tgt"]
        if not s or not t:
            continue
        if max_len is not None and (len(s) > max_len or len(t) > max_len):
            continue
        src.append(s)
        tgt.append(t)
    assert len(src) == len(tgt)
    return src, tgt


def load_split(split, n=None):
    if ON_COLAB:
        ds = load_dataset("wmt/wmt14", "fr-en", split=split, cache_dir=HF_CACHE)
        if n is not None:
            # train is the corpora concatenated one after another; taking the
            # first n would be mostly one corpus, so shuffle the whole split first
            ds = ds.shuffle(seed=DATA_SEED).select(range(n))
        return ds.map(tokenize_pair, remove_columns=["translation"], num_proc=os.cpu_count())
    # Mac: streaming slice (not a random sample), just to exercise the code
    ds = load_dataset("wmt/wmt14", "fr-en", split=split, streaming=True)
    ds = ds.take(n if n is not None else 500)
    return [tokenize_pair(ex) for ex in ds]

# one cache file per n_sample, so the 100k timing cache and the 2M cache coexist
def load_hf_wmt14_data(n_sample=N_SAMPLE):
    cache_path = os.path.join(CACHE_DIR, f"tok_n{n_sample}_len{MAX_LEN}_seed{DATA_SEED}.pt")
    if os.path.exists(cache_path):
        data = torch.load(cache_path)
        print("loaded cache:", cache_path)
    else:
        data = {}
        # train and val get the length filter; test stays complete, since the
        # paper evaluates on all of newstest2014 and Fig. 2 buckets it by length
        data["train_src"], data["train_tgt"] = to_lists(load_split("train", n_sample), MAX_LEN)
        data["val_src"], data["val_tgt"] = to_lists(load_split("validation"), MAX_LEN)  # newstest2013
        data["test_src"], data["test_tgt"] = to_lists(load_split("test"), None)  # newstest2014
        os.makedirs(CACHE_DIR, exist_ok=True)
        torch.save(data, cache_path + ".tmp")
        os.replace(cache_path + ".tmp", cache_path)
        print("saved cache:", cache_path)

    train_src, train_tgt = data["train_src"], data["train_tgt"]
    val_src, val_tgt = data["val_src"], data["val_tgt"]
    test_src, test_tgt = data["test_src"], data["test_tgt"]
    for name, s, t in [("train", train_src, train_tgt), ("val", val_src, val_tgt), ("test", test_src, test_tgt)]:
        assert len(s) == len(t) > 0, name
        print(f"{name}: {len(s)} pairs, max len src {max(map(len, s))} tgt {max(map(len, t))}")
    assert max(map(len, train_src)) <= MAX_LEN and max(map(len, train_tgt)) <= MAX_LEN
    print("example:", train_src[0], "->", train_tgt[0])
    return train_src, train_tgt, val_src, val_tgt, test_src, test_tgt

In [ ]:
# Data Loaders and pre-processors
def build_vocab(tokens, vocab_len=30000):
    tokens = [tok for tok_list in tokens for tok in tok_list]
    token_to_idx = {}
    for tok, idx in [
        (TOK_UNK, TOK_UNK_ID),
        (TOK_EOS, TOK_EOS_ID),
        (TOK_SOS, TOK_SOS_ID),
        (TOK_PAD, TOK_PAD_ID),
    ]:
        token_to_idx[tok] = idx
    tokens_freq = Counter(tokens)
    idx = len(token_to_idx)
    for item, count in tokens_freq.most_common(vocab_len - 4):
        token_to_idx[item] = idx
        idx += 1
    idx_to_token = {idx: token for token, idx in token_to_idx.items()}
    return token_to_idx, idx_to_token


def encode(sentence, token_to_idx):
    unk = token_to_idx[TOK_UNK]
    return [token_to_idx.get(tok, unk) for tok in sentence]


def prepare_dataset(src_sentences, target_sentences, src_vocab, tgt_vocab):
    assert len(src_sentences) == len(target_sentences)
    pairs = []
    for src_sentence, target_sentence in zip(src_sentences, target_sentences):
        pairs.append(
            (encode(src_sentence, src_vocab), encode(target_sentence, tgt_vocab))
        )
    return pairs


def collate(batch):
    src = []
    dec_in = []
    dec_tgt = []
    src_len = []
    for src_tokens, tgt_tokens in batch:
        src_tokens = src_tokens + [TOK_EOS_ID]
        src_len.append(len(src_tokens))
        src_tensor = torch.tensor(src_tokens, dtype=torch.long)
        src.append(src_tensor)
        dec_in_tokens = [TOK_SOS_ID] + tgt_tokens
        dec_tgt_tokens = tgt_tokens + [TOK_EOS_ID]
        dec_in.append(torch.tensor(dec_in_tokens, dtype=torch.long))
        dec_tgt.append(torch.tensor(dec_tgt_tokens, dtype=torch.long))

    def pad_seq(list_of_tokens):
        return torch.nn.utils.rnn.pad_sequence(
            list_of_tokens, batch_first=True, padding_value=TOK_PAD_ID
        )

    src_len = torch.tensor(src_len, dtype=torch.long)
    src = pad_seq(src)
    dec_in = pad_seq(dec_in)
    dec_tgt = pad_seq(dec_tgt)
    assert src.shape[0] == len(batch)
    assert src.shape[1] == src_len.max()
    assert dec_in.shape == dec_tgt.shape
    assert src_len.min() >= 1
    return src, src_len, dec_in, dec_tgt


def prepare_data_loaders(
    src_sentences, target_sentences, src_vocab, tgt_vocab, batch_size=32, shuffle=True
):
    pairs = prepare_dataset(src_sentences, target_sentences, src_vocab, tgt_vocab)
    return DataLoader(pairs, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)  # pyright: ignore[reportArgumentType]  (a list is a valid map-style dataset)

In [ ]:
# Vocab from train only, then train/val loaders. One path for the toy task and WMT14.
# n_train limits only the train loader (e.g. a timing run); the vocab is still
# built from all of train_src/train_tgt, so V matches the full run.
def prepare_data(
    train_src, train_tgt, val_src=None, val_tgt=None, vocab_size=30000, batch_size=32, n_train=None
):
    assert len(train_src) == len(train_tgt)
    src_vocab, src_itos = build_vocab(train_src, vocab_len=vocab_size)
    tgt_vocab, tgt_itos = build_vocab(train_tgt, vocab_len=vocab_size)
    n = len(train_src) if n_train is None else min(n_train, len(train_src))
    train_loader = prepare_data_loaders(
        train_src[:n], train_tgt[:n], src_vocab, tgt_vocab, batch_size=batch_size, shuffle=True
    )
    val_loader = None
    if val_src is not None:
        assert val_tgt is not None and len(val_src) == len(val_tgt)
        val_loader = prepare_data_loaders(
            val_src, val_tgt, src_vocab, tgt_vocab, batch_size=batch_size, shuffle=False
        )
    print(
        f"vocab src {len(src_vocab)}, tgt {len(tgt_vocab)} | train {n} pairs, "
        f"{len(train_loader)} batches of {batch_size}"
        + (f" | val {len(val_src)} pairs" if val_src is not None else "")
    )
    return {
        "src_vocab": src_vocab,
        "tgt_vocab": tgt_vocab,
        "src_itos": src_itos,
        "tgt_itos": tgt_itos,
        "train_loader": train_loader,
        "val_loader": val_loader,
    }

## 4. Model

In [ ]:
# Encoder
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden_dim):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.enc_hidden_dim = enc_hidden_dim
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=TOK_PAD_ID)
        self.rnn = nn.GRU(
            input_size=emb_dim,
            hidden_size=enc_hidden_dim,
            bidirectional=True,
            batch_first=True,
        )

    def forward(self, src: torch.Tensor, src_len: torch.Tensor):
        """
        src shape : (batch_size, max_src_len)
        src_len : (batch_size,)
        """
        # 1. shape (batch_size, src_len, emb_dim)
        embedd = self.emb(src)
        packed_embedd = pack_padded_sequence(
            embedd, src_len.cpu(), batch_first=True, enforce_sorted=False
        )
        # packed_outputs shape (batch_size, max_src_len, 2*enc_hidden_dim)
        # hidden = (2, batch_size, enc_hidden_dim)
        packed_outputs, hidden = self.rnn(packed_embedd)
        encoder_outputs, _ = pad_packed_sequence(
            packed_outputs, batch_first=True, total_length=src.shape[1]
        )
        assert encoder_outputs.shape == (
            src.shape[0],
            src.shape[1],
            2 * self.enc_hidden_dim,
        )
        # hidden[0] = h_fwd_T (fwd state at each real last token, thanks to packing)
        # hidden[1] = h_bwd_1 (bwd state at position 1)
        assert hidden.shape == (2, src.shape[0], self.enc_hidden_dim)

        # decoder initial_hidden_state shape (batch_size, enc_hidden_dim)
        mask_src = src != TOK_PAD_ID
        assert mask_src.shape == (src.shape[0], src.shape[1])
        return encoder_outputs, hidden, mask_src

In [ ]:
# Attention
class Attention(nn.Module):
    def __init__(self, enc_hidden_dim, dec_hidden_dim, attn_dim):
        super().__init__()
        self.V = nn.Linear(attn_dim, 1, bias=True)
        self.W = nn.Linear(dec_hidden_dim, attn_dim, bias=True)
        self.U = nn.Linear(2 * enc_hidden_dim, attn_dim, bias=True)

    def precompute_keys(self, H_enc):
        return self.U(H_enc)

    def forward(self, s_prev, keys, H_enc, mask):
        s_prev_wa = self.W(s_prev)
        s_prev_wa = s_prev_wa.unsqueeze(1)
        e = self.V(torch.tanh(s_prev_wa + keys)).squeeze(-1)
        assert e.shape == (H_enc.shape[0], H_enc.shape[1])
        e = e.masked_fill(~mask, float("-inf"))
        attn_weights = torch.softmax(e, dim=1)
        assert attn_weights.shape == (H_enc.shape[0], H_enc.shape[1])
        context = torch.bmm(attn_weights.unsqueeze(1), H_enc)
        context = context.squeeze(1)
        assert context.shape == (H_enc.shape[0], H_enc.shape[2])
        return context, attn_weights

In [ ]:
# Decoder: one step
class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        emb_dim,
        dec_hidden_dim,
        enc_hidden_dim,
        attn_dim,
        max_out_dim,
        use_attention=True,
    ):
        super().__init__()
        self.emb_dim = emb_dim
        self.dec_hidden_dim = dec_hidden_dim
        self.enc_hidden_dim = enc_hidden_dim
        self.attn_dim = attn_dim
        self.max_out_dim = max_out_dim
        self.vocab_size = vocab_size
        self.use_attention = use_attention
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=TOK_PAD_ID)
        # baseline (use_attention=False): no attention module at all
        self.attention = (
            Attention(
                enc_hidden_dim=enc_hidden_dim,
                dec_hidden_dim=dec_hidden_dim,
                attn_dim=attn_dim,
            )
            if use_attention
            else None
        )
        self.rnn = nn.GRUCell(
            input_size=emb_dim + (2 * enc_hidden_dim), hidden_size=dec_hidden_dim
        )
        self.bridge = nn.Linear(enc_hidden_dim, dec_hidden_dim)
        self.fc_out = nn.Linear(
            emb_dim + dec_hidden_dim + 2 * enc_hidden_dim, 2 * max_out_dim
        )
        self.W_o = nn.Linear(max_out_dim, vocab_size)

    def init_state(self, h_bwd_1):
        return torch.tanh(self.bridge(h_bwd_1))

    def forward(self, y_prev, s_prev, keys, H_enc, mask, c_fixed=None):
        """
        use_attention=True : c_i from attention, recomputed every step (keys, H_enc, mask used)
        use_attention=False: c_fixed = [h_fwd_T ; h_bwd_1], the same vector every step
        """
        # emb shape (B, emb_dim)
        emb = self.emb(y_prev)
        assert emb.shape == (y_prev.shape[0], self.emb_dim)
        # context shape (B, 2*enc_hidden_dim)
        # attn_weights (B, Tin), or None for the baseline
        if self.attention is not None:
            context, attn_weights = self.attention(s_prev, keys, H_enc, mask)
            assert attn_weights.shape == (y_prev.shape[0], H_enc.shape[1])
        else:
            assert c_fixed is not None, "baseline decoder needs c_fixed"
            context, attn_weights = c_fixed, None
        assert context.shape == (y_prev.shape[0], 2 * self.enc_hidden_dim)
        # rnn_input shape (B, emb_dim + 2*enc_hidden_dim)
        rnn_input = torch.cat((emb, context), dim=1)
        assert rnn_input.shape == (
            y_prev.shape[0],
            self.emb_dim + 2 * self.enc_hidden_dim,
        )
        # s_curr shape (B, dec_hidden_dim)
        s_curr = self.rnn(
            rnn_input,  # (B, emb_dim + 2*enc_hidden_dim)
            s_prev,  # (B, dec_hidden_dim)
        )
        assert s_curr.shape == (y_prev.shape[0], self.dec_hidden_dim)
        combined = torch.cat((emb, s_curr, context), dim=1)
        t_tilde = self.fc_out(combined)
        t = t_tilde.view(y_prev.shape[0], self.max_out_dim, 2)
        t = t.max(dim=2).values
        assert t.shape == (y_prev.shape[0], self.max_out_dim)
        logits = self.W_o(t)
        assert logits.shape == (y_prev.shape[0], self.vocab_size)
        return logits, s_curr, attn_weights

In [ ]:
# Seq2Seq: teacher-forced forward and greedy predict
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def _init_values(self, src, src_len):
        H_enc, hidden, mask = self.encoder(src, src_len)
        h_fwd_T, h_bwd_1 = hidden[0], hidden[1]  # (B, H) each
        s = self.decoder.init_state(h_bwd_1)  # s_0 from h_bwd_1 in both models
        if self.decoder.use_attention:
            keys = self.decoder.attention.precompute_keys(H_enc)
            c_fixed = None
        else:
            keys = None
            c_fixed = torch.cat((h_fwd_T, h_bwd_1), dim=1)
            assert c_fixed.shape == (src.shape[0], 2 * self.encoder.enc_hidden_dim)
        return H_enc, s, keys, mask, c_fixed

    def forward(self, src, src_len, dec_in):
        H_enc, s, keys, mask, c_fixed = self._init_values(src, src_len)
        T_out = dec_in.shape[1]
        logits = []
        attn_weights = []
        for i in range(T_out):
            logits_i, s, a_i = self.decoder(
                dec_in[:, i], s, keys, H_enc, mask, c_fixed=c_fixed
            )
            logits.append(logits_i)
            attn_weights.append(a_i)
        logits = torch.stack(logits, dim=1)
        assert logits.shape == (src.shape[0], T_out, self.decoder.vocab_size)
        if not self.decoder.use_attention:
            return logits, None
        attn_weights = torch.stack(attn_weights, dim=1)
        assert attn_weights.shape == (src.shape[0], T_out, src.shape[1])
        return logits, attn_weights

    @torch.no_grad
    def predict(self, src, src_len, max_len):
        device = src.device
        batch_size, T_in = src.shape
        H_enc, s, keys, mask, c_fixed = self._init_values(src, src_len)
        y_prev = torch.full((batch_size,), TOK_SOS_ID, dtype=torch.long, device=device)
        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
        predictions = []
        attn_weights = []
        for _ in range(max_len):
            logits_i, s, a_i = self.decoder(
                y_prev, s, keys, H_enc, mask, c_fixed=c_fixed
            )
            y_pred = torch.argmax(logits_i, dim=1)
            # sentences that already emitted EOS get PAD from here on
            y_pred = y_pred.masked_fill(finished, TOK_PAD_ID)
            assert y_pred.shape == (batch_size,)
            predictions.append(y_pred)
            if a_i is not None:
                assert a_i.shape == (batch_size, T_in)
                attn_weights.append(a_i)
            finished |= y_pred == TOK_EOS_ID
            y_prev = y_pred
            if finished.all():
                break
        predictions = torch.stack(predictions, dim=1)
        T = predictions.shape[1]
        assert predictions.shape == (batch_size, T)
        if not self.decoder.use_attention:
            return predictions, None
        attn_weights = torch.stack(attn_weights, dim=1)
        assert attn_weights.shape == (batch_size, T, T_in)
        return predictions, attn_weights

In [ ]:
# Build RNNsearch (use_attention=True) or the RNNencdec-style baseline (False)
def build_model(src_vocab_size, tgt_vocab_size, emb, H, A, max_out_dim, use_attention):
    encoder = Encoder(vocab_size=src_vocab_size, emb_dim=emb, enc_hidden_dim=H)
    decoder = Decoder(
        vocab_size=tgt_vocab_size,
        emb_dim=emb,
        enc_hidden_dim=H,
        dec_hidden_dim=H,
        attn_dim=A,
        max_out_dim=max_out_dim,
        use_attention=use_attention,
    )
    model = Seq2Seq(encoder=encoder, decoder=decoder)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"model: use_attention={use_attention}, {n_params:,} params")
    return model

## 5. Training

In [ ]:
def make_batch_loss_fn(vocab_size: int):
    criterion = nn.CrossEntropyLoss(ignore_index=TOK_PAD_ID)

    def batch_loss_fn(
        model: nn.Module, batch: Any, dev: torch.device
    ) -> tuple[torch.Tensor, float]:
        src, src_len, dec_in, dec_tgt = [x.to(dev) for x in batch]
        logits, _ = model(src, src_len, dec_in)
        loss = criterion(logits.reshape(-1, vocab_size), dec_tgt.reshape(-1))
        n_tokens = (dec_tgt != TOK_PAD_ID).sum().item()
        return loss, float(n_tokens)

    return batch_loss_fn


def train(
    model: nn.Module,
    tgt_vocab_size: int,
    train_data_loader: DataLoader[Any],
    val_data_loader: DataLoader[Any] | None,
    epochs: int,
    rho: float = 0.95,
    eps: float = 1e-6,
    experiment_name: str | None = None,
    save_every_steps: int | None = None,
    run_config: dict[str, Any] | None = None,
    vocab: dict[str, Any] | None = None,
    profile: bool = False,
) -> Trainer:
    optimizer = torch.optim.Adadelta(model.parameters(), rho=rho, eps=eps)
    trainer = Trainer(
        model=model,
        optimizer=optimizer,
        batch_loss_fn=make_batch_loss_fn(tgt_vocab_size),
        max_grad_norm=1.0,
        device=device,
        experiment_name=experiment_name,
        save_every_steps=save_every_steps,
        run_config=run_config,
        vocab=vocab,
        seed=seed,
        profile=profile,
    )
    trainer.fit(train_data_loader, val_data_loader, epochs=epochs)
    return trainer


# data from prepare_data, model_cfg = build_model kwargs (emb, H, A, max_out_dim, use_attention).
# run_config holds everything that defines the run, so the checkpointer refuses
# to resume a different setup; epochs is left out so a run can be extended.
def run_experiment(experiment_name, data, model_cfg, epochs, save_every_steps=None, profile=False):
    model = build_model(len(data["src_vocab"]), len(data["tgt_vocab"]), **model_cfg)
    run_config = {
        **model_cfg,
        "src_vocab_size": len(data["src_vocab"]),
        "tgt_vocab_size": len(data["tgt_vocab"]),
        "n_train": len(data["train_loader"].dataset),
        "batch_size": data["train_loader"].batch_size,
    }
    trainer = train(
        model=model,
        tgt_vocab_size=len(data["tgt_vocab"]),
        train_data_loader=data["train_loader"],
        val_data_loader=data["val_loader"],
        epochs=epochs,
        experiment_name=experiment_name,
        save_every_steps=save_every_steps,
        run_config=run_config,
        vocab={"src": data["src_vocab"], "tgt": data["tgt_vocab"]},
        profile=profile,
    )
    return trainer

## 6. Evaluation

In [ ]:
# Greedy per-token accuracy: by sentence-length bucket (Fig. 2 view) and by output position
def token_accuracy(model, data_loader, max_decode_len, bucket_size=5):
    """
    Greedy-decode every batch with model.predict and compare to dec_tgt position by position.

    Counting rule (per sentence, target = words + EOS):
      - denominator = number of real target tokens (dec_tgt != PAD)
      - a missing position (model emitted EOS early) counts as wrong
      - extra positions (model ran past the target's EOS) are ignored; that miss is
        already counted once, at the target's EOS position

    Returns:
      by_bucket : {(lo, hi): (accuracy, n_target_tokens, n_sentences)}, bucketed by the
                  number of source words, e.g. (6, 10), (11, 15), ...
      by_pos    : tensor (max_decode_len,), accuracy at output step t (0-indexed);
                  NaN where no sentence has a target token at that step
      exact     : fraction of sentences with every target token correct
    """
    model.eval()
    pos_correct = torch.zeros(max_decode_len)
    pos_total = torch.zeros(max_decode_len)
    bucket_correct, bucket_total, bucket_sents = {}, {}, {}
    n_exact, n_sents = 0, 0

    for src, src_len, _, dec_tgt in data_loader:
        src, src_len, dec_tgt = src.to(device), src_len.to(device), dec_tgt.to(device)
        batch_size, T_tgt = dec_tgt.shape
        assert T_tgt <= max_decode_len, "max_decode_len must cover the longest target"

        pred, _ = model.predict(src, src_len, max_len=max_decode_len)
        T_pred = pred.shape[1]
        # align widths: PAD-fill a shorter prediction, cut a longer one
        if T_pred < T_tgt:
            pred = torch.nn.functional.pad(pred, (0, T_tgt - T_pred), value=TOK_PAD_ID)
        pred = pred[:, :T_tgt]
        assert pred.shape == dec_tgt.shape == (batch_size, T_tgt)

        tgt_mask = dec_tgt != TOK_PAD_ID  # real target positions (words + EOS)
        # PAD never equals a real target token, so missing positions count as wrong
        hit = (pred == dec_tgt) & tgt_mask
        assert hit.shape == (batch_size, T_tgt)

        pos_correct[:T_tgt] += hit.sum(dim=0).float().cpu()
        pos_total[:T_tgt] += tgt_mask.sum(dim=0).float().cpu()

        sent_correct = hit.sum(dim=1)  # (B,)
        sent_total = tgt_mask.sum(dim=1)  # (B,)
        n_exact += int((sent_correct == sent_total).sum())
        n_sents += batch_size

        n_words = (src_len - 1).tolist()  # src = words + EOS
        for w, c, t in zip(n_words, sent_correct.tolist(), sent_total.tolist()):
            lo = ((w - 1) // bucket_size) * bucket_size + 1  # 6..10 -> 6, 11..15 -> 11
            key = (lo, lo + bucket_size - 1)
            bucket_correct[key] = bucket_correct.get(key, 0) + c
            bucket_total[key] = bucket_total.get(key, 0) + t
            bucket_sents[key] = bucket_sents.get(key, 0) + 1

    by_bucket = {
        k: (bucket_correct[k] / bucket_total[k], bucket_total[k], bucket_sents[k])
        for k in sorted(bucket_total)
    }
    by_pos = torch.where(pos_total > 0, pos_correct / pos_total.clamp(min=1), torch.nan)
    exact = n_exact / n_sents
    return by_bucket, by_pos, exact

In [ ]:
# Load a trained model from its best.pt; data and model_cfg must match the run
def load_best(experiment_name, data, model_cfg):
    m = build_model(len(data["src_vocab"]), len(data["tgt_vocab"]), **model_cfg).to(device)
    loaded = Checkpointer(experiment_name).load(CheckpointState(model=m), tag="best", device=device)
    assert loaded, f"no best.pt for {experiment_name}"
    return m.eval()


# Token accuracy by source-length bucket for several models on one loader.
# models: {name: model}. Returns {name: (by_bucket, by_pos, exact)}.
def compare_models(models, data_loader, max_decode_len):
    results = {}
    for name, m in models.items():
        by_bucket, by_pos, exact = token_accuracy(m, data_loader, max_decode_len=max_decode_len)
        results[name] = (by_bucket, by_pos, exact)
        print(f"{name}: exact match {exact:.3f}")
        for (lo, hi), (acc, n_tok, n_sent) in by_bucket.items():
            print(f"  len {lo:>2}-{hi:<2}  token acc {acc:.3f}  ({n_sent} sentences, {n_tok} tokens)")
    return results

In [ ]:
# Attention heatmap: rows = output steps, columns = source tokens
def plot_attention_heatmap(src_words, tgt_words, attn_matrix):
    fig, ax = plt.subplots(figsize=(7, 6))
    cax = ax.matshow(attn_matrix, cmap="bone")
    fig.colorbar(cax)

    ax.set_xticks(range(len(src_words)))
    ax.set_yticks(range(len(tgt_words)))
    ax.set_xticklabels(src_words, rotation=45, ha="left", fontsize=11)
    ax.set_yticklabels(tgt_words, fontsize=11)

    for i in range(len(tgt_words)):
        for j in range(len(src_words)):
            val = attn_matrix[i, j]
            color = "black" if val > 0.5 else "white"
            ax.text(
                j, i, f"{val:.2f}", ha="center", va="center", color=color, fontsize=9
            )

    ax.set_xlabel("Source Sentence (Tin)", fontsize=12)
    ax.set_ylabel("Predicted Target (Tout)", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Table 1 counter: tokens, types, UNK for one segmentation. TODO (Vikas)
def table1_row(train_sentences, test_sentences):
    """Inputs are already segmented (list of token lists). Returns the Table 1 numbers."""
    raise NotImplementedError

## 7. Experiment: Table 1 on WMT14 French (whole words, characters, BPE)

Train: cached WMT14 En-Fr train (~1.65M pairs, French side). UNK on newstest2014 (cached `test`, complete). The paper uses newstest2013.

In [ ]:
# TODO (Vikas)

## 8. Experiment: RNNsearch-50 with BPE on WMT14 En-Fr

In [ ]:
# TODO (Vikas)